In [ ]:
%pip install -q -U langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu wikipedia arxiv beautifulsoup4 python-dotenv


In [6]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [16]:
api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=200)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper)

In [17]:
wiki.name

'wikipedia'

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents, OpenAIEmbeddings(model="text-embedding-3-small"))
retriever = vectordb.as_retriever()
retriever


In [ ]:
from langchain_core.tools import create_retriever_tool

retriever_tool = create_retriever_tool(retriever, "langsmith_search",
                      "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!")


In [14]:
retriever_tool.name

'langsmith_search'

In [15]:
## Arxiv Tool
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper=ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=200)
arxiv=ArxivQueryRun(api_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [18]:
tools=[wiki,arxiv,retriever_tool]

In [19]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'e:\\Langchain\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=200, arxiv_result=<class 'arxiv.Result'>)),
 Tool(name='langsmith_search', description='Search for information about LangSmith. For any questions about LangSmith, you must use this tool!', args_schema=<class 'langchain.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x0000023F870F6290>, retriever=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorsto

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to your .env file"

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-6-luna"), temperature=0)


In [ ]:
# The LangChain Hub prompt is no longer needed: create_agent takes a system prompt directly.
system_prompt = "You are a helpful assistant. Use the available tools to answer questions."


In [ ]:
### Agents (LangChain 1.x)
from langchain.agents import create_agent

agent = create_agent(llm, tools, system_prompt=system_prompt)


In [ ]:
## Helper to ask the agent a question and print the final answer
def ask(question):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    for message in result["messages"]:
        message.pretty_print()
    return result["messages"][-1].content


In [ ]:
ask("Tell me about Langsmith")


In [ ]:
ask("What's the paper 1605.08386 about?")
